In [0]:
# Cell 1
%pip install langgraph langchain langchain-community --quiet
%pip install --upgrade databricks-langchain langchain-community langchain databricks-sql-connector


In [0]:
# %%writefile agent_complete_with_chart.py
import json
import re
from uuid import uuid4
from typing import Generator, List, Optional
from dataclasses import dataclass
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client
from langgraph.graph import StateGraph, END
from langgraph.graph.state import CompiledStateGraph
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest, ResponsesAgentResponse, ResponsesAgentStreamEvent,
    output_to_responses_items_stream, to_chat_completions_input,
)

# Setup LLM and Databricks client
client = DatabricksFunctionClient()
set_uc_function_client(client)

@dataclass
class ToolCall:
    tool: str
    input: dict
    agent: str
    reasoning: str

@dataclass
class ToolResult:
    tool: str
    output: any
    success: bool
    error: Optional[str]
    metadata: dict

class ToolExecutor:
    def __init__(self, spark, llm, table_name: str, max_rows: int = 2000):
        self.spark = spark
        self.llm = llm
        self.table_name = table_name
        self.max_rows = max_rows
    
    def execute(self, tool_call: ToolCall) -> ToolResult:
        try:
            if tool_call.tool == "sql_validate":
                return self._validate_sql(tool_call.input)
            if tool_call.tool == "sql_execute":
                return self._execute_sql(tool_call.input)
            if tool_call.tool == "data_profile":
                return self._profile_data(tool_call.input)
            if tool_call.tool == "plot_render":
                return self._render_plot(tool_call.input)
            return ToolResult(tool=tool_call.tool, output=None, success=False, error="Unknown tool", metadata={})
        except Exception as e:
            return ToolResult(tool=tool_call.tool, output=None, success=False, error=str(e), metadata={})
    
    def _validate_sql(self, input_dict: dict) -> ToolResult:
        query = input_dict["query"]
        if "SELECT" not in query.upper():
            return ToolResult("sql_validate", None, False, "No SELECT in query", {})
        forbidden = ["DROP", "DELETE", "TRUNCATE", "ALTER"]
        if any(kw in query.upper() for kw in forbidden):
            return ToolResult("sql_validate", None, False, "Destructive statement forbidden", {})
        return ToolResult("sql_validate", {"valid": True}, True, None, {})
    
    def _execute_sql(self, input_dict: dict) -> ToolResult:
        query = input_dict["query"]
        val = self._validate_sql(input_dict)
        if not val.success:
            return ToolResult("sql_execute", None, False, val.error, {})
        df = self.spark.sql(query).limit(self.max_rows).toPandas()
        output = {"dataframe": df, "rows": json.loads(df.to_json(orient="records")), "columns": list(df.columns), "dtypes": {col:str(dtype) for col, dtype in df.dtypes.items()}}
        return ToolResult("sql_execute", output, True, None, {"rows_fetched": len(df)})
    
    def _profile_data(self, input_dict: dict) -> ToolResult:
        df = input_dict.get("dataframe")
        if df is None or df.empty:
            return ToolResult("data_profile", None, False, "Empty dataframe", {})
        profile = {
            "null_counts": df.isnull().sum().to_dict(),
            "unique_counts": {c: df[c].nunique() for c in df.columns},
            "shape": df.shape
        }
        return ToolResult("data_profile", profile, True, None, {})
    
    def _render_plot(self, input_dict: dict) -> ToolResult:
        code = input_dict.get("code", "")
        try:
            compile(code, "<chart_code>", "exec")
            return ToolResult("plot_render", {"compiled": True}, True, None, {})
        except Exception as e:
            return ToolResult("plot_render", None, False, str(e), {})

class CustomAgent:
    def __init__(self, name: str, llm, tool_executor: ToolExecutor):
        self.name = name
        self.llm = llm
        self.tool_executor = tool_executor
        self.max_iter = 3
    
    def call_llm(self, prompt: str) -> str:
        try:
            if hasattr(self.llm, "invoke"):
                out = self.llm.invoke(prompt)
            else:
                out = self.llm(prompt)
            if isinstance(out, str):
                return out
            if hasattr(out, "content"):
                return out.content
            if hasattr(out, "text"):
                return out.text
            return str(out)
        except Exception as e:
            print(f"LLM call error: {e}")
            return ""
    
    def think(self, state: dict) -> str:
        prompt = f"Agent: {self.name}\nQuestion: {state.get('question', '')}\nNext action?"
        return self.call_llm(prompt)
    
    def act(self, state: dict, thought: str) -> List[ToolResult]:
        return []
    
    def run(self, state: dict) -> dict:
        for _ in range(self.max_iter):
            thought = self.think(state)
            results = self.act(state, thought)
            if all(r.success for r in results):
                break
            state.setdefault("tool_results", []).extend(results)
        return state

class SQLGeneratorAgent(CustomAgent):
    def __init__(self, llm, tool_executor, table_name):
        super().__init__("SQLGeneratorAgent", llm, tool_executor)
        self.table_name = table_name

    def act(self, state: dict, thought: str) -> List[ToolResult]:
        question = state.get("question", "")
        prompt = f"""You are an expert SQL generator for Databricks Delta tables.
Available table: `{self.table_name}` with columns:
DATE, ZONE, REGION, COUNTRY, RETAIL_CHANNEL, RETAILER_NAME, MANUFACTURER, PRODUCT_FAMILY, SPECIES, BRAND, SUB_BRAND, SKU_NAME, MARKETING_CHANNEL, CAMPAIGN_NAME, METRIC, VALUE
Rules:
- Return a single VALID Databricks SQL SELECT statement, no explanation, no markdown fences.
- Use uppercase for SQL keywords.
- Use ISO date format 'YYYY-MM-DD'.
- Always include an ORDER BY.
- Do NOT include destructive statements.
User question: {question}"""
        raw_sql = self.call_llm(prompt)
        sql_query = re.search(r"(?i)(select\b[\s\S]*)", raw_sql)
        sql_query = sql_query.group(1).strip() if sql_query else raw_sql.strip()
        val = self.tool_executor.execute(ToolCall("sql_validate", {"query": sql_query}, self.name, thought))
        results = [val]
        if val.success:
            exec_result = self.tool_executor.execute(ToolCall("sql_execute", {"query": sql_query}, self.name, "Execute query"))
            results.append(exec_result)
            if exec_result.success:
                state["generated_sql"] = sql_query
                state["dataframe"] = exec_result.output["dataframe"]
                state["schema"] = {"columns": exec_result.output["columns"], "dtypes": exec_result.output["dtypes"]}
        return results

class DataAnalysisAgent(CustomAgent):
    def __init__(self, llm, tool_executor):
        super().__init__("DataAnalysisAgent", llm, tool_executor)

    def act(self, state: dict, thought: str) -> List[ToolResult]:
        df = state.get("dataframe")
        if df is None or df.empty:
            return []
        profile_result = self.tool_executor.execute(ToolCall("data_profile", {"dataframe": df}, self.name, "Profile data"))
        results = [profile_result]
        question = state.get("question", "")
        schema_str = json.dumps(state.get("schema", {}))
        sample_json = json.dumps(df.head(5).to_dict(orient="records"), default=str)
        analysis_prompt = f"""You are a data analysis expert. Provide key insights and recommendations:
Schema: {schema_str}
Sample data: {sample_json}
User question: {question}
Format: Plain text only."""
        analysis_text = self.call_llm(analysis_prompt)
        state["analysis_text"] = analysis_text
        results.append(ToolResult("text_analysis", {"text": analysis_text}, True, None, {}))
        return results

class ChartGeneratorAgent(CustomAgent):
    def __init__(self, llm, tool_executor):
        super().__init__("ChartGeneratorAgent", llm, tool_executor)

    def act(self, state: dict, thought: str) -> List[ToolResult]:
        df = state.get("dataframe")
        if df is None or df.empty:
            return []

        # profile the dataframe (keeps existing behavior)
        profile_result = self.tool_executor.execute(ToolCall("data_profile", {"dataframe": df}, self.name, "Profile data for chart"))
        results = [profile_result]

        question = state.get("question", "")
        sample_json = json.dumps(df.head(5).to_dict(orient="records"), default=str)

        # Strict plan prompt: force using exact sample columns and forbid creating new data
        plan_prompt = (
            "You are a visualization planner. Return ONLY a single JSON object (no prose). "
            "Use exact column names from the sample (case-sensitive). DO NOT invent or fabricate columns. "
            "DO NOT create or return a DataFrame. The chart must be based on the existing dataframe variable `df`.\n\n"
            "Return JSON with keys: chart_type (LINE|BAR|PIE|HEATMAP), x (column or null), y (list of 1-2 columns), "
            "aggregation (sum|avg|count|null), title (short string).\n\n"
            f"Sample data: {sample_json}\n"
            f"Question: {question}\n"
        )

        # Ask LLM for plan
        plan_json = self.call_llm(plan_prompt)
        # Parse plan safely and repair using actual df columns if needed
        try:
            cleaned_json_text = re.sub(r"^```(?:json)?|```$", "", plan_json, flags=re.IGNORECASE).strip()
            plan = json.loads(cleaned_json_text)
        except Exception:
            # conservative default plan derived from actual df columns (never fabricate)
            plan = {"chart_type": "LINE", "x": None, "y": list(df.columns)[:2], "aggregation": "sum", "title": question[:80]}

        # Repair/validate plan to ensure referenced columns actually exist in df
        def _valid_col(col):
            return col is not None and isinstance(col, str) and col in list(df.columns)

        # Normalize chart_type
        if plan.get("chart_type") not in {"LINE", "BAR", "PIE", "HEATMAP"}:
            plan["chart_type"] = "LINE" if "DATE" in df.columns else "BAR"

        # Validate x
        if not _valid_col(plan.get("x")):
            if "DATE" in df.columns:
                plan["x"] = "DATE"
            else:
                non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
                plan["x"] = non_numeric[0] if non_numeric else (list(df.columns)[0] if len(df.columns) else None)

        # Validate y (must be actual columns)
        y_candidates = [c for c in (plan.get("y") or []) if c in df.columns]
        if not y_candidates:
            numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != plan.get("x")]
            y_candidates = numeric[:2] if numeric else [c for c in df.columns if c != plan.get("x")][:2]
        plan["y"] = y_candidates

        # Validate aggregation
        if plan.get("aggregation") not in {"sum", "avg", "count", None}:
            plan["aggregation"] = "sum"

        # Build a strict code prompt that enforces use of `df` and forbids creating new data
        code_prompt = (
            "You are a Python/Plotly code generator. Produce ONLY executable Python code (no prose, no fences). "
            "IMPORTANT RULES:\n"
            "- The code MUST use the existing pandas DataFrame variable named `df` (this is the full query result).\n"
            "- DO NOT create a new DataFrame or hardcode sample values.\n"
            "- DO NOT reference columns that are not present in `df`.\n"
            "- The code MUST assign a Plotly figure to variable `fig` and call `fig.show()` as the last statement.\n"
            "- Do NOT include import statements for pandas/plotly (pd/px/go are provided by runtime).\n"
            "- Use defensive checks like `if 'COL' in df.columns` before using a column.\n\n"
            f"Plan JSON: {json.dumps(plan)}\n"
            f"Question: {question}\n"
            f"Sample columns: {list(df.columns)}\n"
        )

        # Request code from LLM
        code = self.call_llm(code_prompt)

        # Strip code fences and top-level import lines (we provide pd/px/go)
        code = re.sub(r"^```(?:python)?\s*", "", code, flags=re.IGNORECASE).strip()
        code = re.sub(r"\s*```\s*$", "", code, flags=re.IGNORECASE)
        code_lines = [ln for ln in code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
        cleaned_code = "\n".join(code_lines).strip()

        # Quick static checks: must reference df, must define fig, must call fig.show
        if ("df" not in cleaned_code) or ("fig" not in cleaned_code) or ("fig.show" not in cleaned_code):
            # do not accept code that creates its own DataFrame or doesn't operate on df
            validation = ToolResult(tool="plot_render", output=None, success=False, error="Chart code missing required references (df/fig/fig.show).", metadata={})
            results.append(validation)
            return results

        # Attempt to compile to catch syntax errors before remote validation
        try:
            compile(cleaned_code, "<chart_code>", "exec")
        except Exception as e:
            validation = ToolResult(tool="plot_render", output=None, success=False, error=f"Chart code compilation error: {e}", metadata={})
            results.append(validation)
            return results

        # Ask the tool_executor to validate/render the chart (existing behavior)
        validation = self.tool_executor.execute(ToolCall("plot_render", {"code": cleaned_code, "dataframe": df}, self.name, "Validate chart code"))
        results.append(validation)

        if validation.success:
            # store the cleaned code (which uses real df) into state for downstream execution
            state["chart_code"] = cleaned_code
        else:
            # if validation failed, do not set chart_code (strict mode, no fallback)
            state["chart_code"] = ""

        return results


def create_custom_agent_langgraph(llm, tool_executor, table_name="demo.retail_media"):
    sql_agent = SQLGeneratorAgent(llm, tool_executor, table_name)
    analysis_agent = DataAnalysisAgent(llm, tool_executor)
    chart_agent = ChartGeneratorAgent(llm, tool_executor)

    def supervisor_node(state: dict) -> dict:
        messages = state.get("messages", [])
        if not messages:
            return state
        last_msg = messages[-1].get("content", "")
        state["question"] = last_msg
        return state

    def sql_node(state: dict) -> dict:
        state["tool_results"] = []
        return sql_agent.run(state)

    def analysis_node(state: dict) -> dict:
        return analysis_agent.run(state)

    def chart_node(state: dict) -> dict:
        return chart_agent.run(state)

    graph = StateGraph(dict)
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("sql_agent", sql_node)
    graph.add_node("analysis_agent", analysis_node)
    graph.add_node("chart_agent", chart_node)

    graph.add_edge("supervisor", "sql_agent")
    graph.add_edge("sql_agent", "analysis_agent")
    graph.add_edge("analysis_agent", "chart_agent")
    graph.add_edge("chart_agent", END)

    graph.set_entry_point("supervisor")
    return graph.compile()

class SimpleAgent:
    def __init__(self, graph: CompiledStateGraph):
        self.graph = graph
    
    def run(self, question: str) -> dict:
        input_state = {
            "messages": [{"content": question}],
            "question": question
        }
        final_state = None
        for step in self.graph.stream(input_state):
            final_state = step

        # ---- Minimal addition: execute and visualize chart_code (if present) ----
        try:
            if final_state and isinstance(final_state, dict):
                chart_code = final_state.get("chart_code", "") or final_state.get("chart_code_snippet", "")
                df = final_state.get("dataframe") or final_state.get("df")
                # Only attempt execution if we have code and a real dataframe
                if chart_code and isinstance(chart_code, str) and chart_code.strip() and df is not None and isinstance(df, pd.DataFrame) and not df.empty:
                    # Remove any top-level imports (runtime already provides pd/px/go)
                    cleaned_lines = [ln for ln in chart_code.splitlines() if not re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln)]
                    cleaned_code = "\n".join(cleaned_lines).strip()
                    # Prepare safe execution environment (provide pd, px, go, df)
                    local_env = {"pd": pd, "px": px, "go": go, "df": df.copy(), "__builtins__": {"len": len, "range": range, "min": min, "max": max, "sum": sum}}
                    try:
                        compiled = compile(cleaned_code, "<chart_exec>", "exec")
                        exec(compiled, local_env)
                        fig = local_env.get("fig")
                        if fig is not None:
                            try:
                                fig.show()
                            except Exception:
                                try:
                                    # fallback to display if available in notebook env
                                    from IPython.display import display
                                    display(fig)
                                except Exception:
                                    print("Chart produced but could not be displayed.")
                        else:
                            print("Chart code executed but no 'fig' was produced.")
                    except Exception as e:
                        print("Error executing chart code:", e)
        except Exception as e:
            print("Error during post-run chart execution:", e)
        # ---- end addition ----

        return final_state or input_state

# Initialization
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
TABLE_NAME = "demo.retail_media"
MAX_SQL_ROWS = 2000

print("Setting up LLM and executor...")
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
tool_executor = ToolExecutor(spark, llm, TABLE_NAME, MAX_SQL_ROWS)
compiled_graph = create_custom_agent_langgraph(llm, tool_executor, TABLE_NAME)
agent = SimpleAgent(compiled_graph)
print("Setup complete. Ready to run.")

# To run:
# result = agent.run("Your NL question here")
# print(result.get("analysis_text"))
# print(result.get("chart_code"))
# result['dataframe'].head()


In [0]:
import json, re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def present(agent_result):
    # -----------------------
    # Normalize JSON input
    # -----------------------
    if isinstance(agent_result, str):
        try:
            agent_result = json.loads(agent_result)
        except Exception:
            raise ValueError("agent_result string is not valid JSON")

    # If wrapped like {"chart_agent": {...}}
    core = agent_result.get("chart_agent", agent_result)

    # -----------------------
    # Extract fields safely
    # -----------------------
    analysis = core.get("analysis_text") or core.get("analysis") or None
    sql = core.get("generated_sql") or core.get("sql") or None
    df_obj = core.get("dataframe")
    chart_code = core.get("chart_code") or core.get("chart") or core.get("chart_code_snippet") or None

    # -----------------------
    # 1) Print analysis
    # -----------------------
    print("\n" + "="*40)
    print("DATA ANALYSIS")
    print("="*40)
    print(analysis if analysis else "<no analysis text found>")

    # -----------------------
    # 2) Print SQL
    # -----------------------
    print("\n" + "="*40)
    print("GENERATED SQL")
    print("="*40)
    print(sql if sql else "<no generated SQL found>")

    # -----------------------
    # 3) Convert dataframe
    # -----------------------
    df = None

    if isinstance(df_obj, pd.DataFrame):
        df = df_obj

    elif isinstance(df_obj, list):
        try:
            df = pd.DataFrame(df_obj)
        except Exception:
            df = None

    elif isinstance(df_obj, dict):
        if "rows" in df_obj and isinstance(df_obj["rows"], list):
            df = pd.DataFrame(df_obj["rows"])
        else:
            try:
                df = pd.DataFrame(df_obj)
            except Exception:
                df = None

    # -----------------------
    # Print dataframe
    # -----------------------
    print("\n" + "="*40)
    print("DATAFRAME (top rows)")
    print("="*40)

    if df is None:
        print("<no dataframe found or could not convert>")
    else:
        print(f"Shape: {df.shape}")
        try:
            from IPython.display import display
            display(df.head(10))
        except Exception:
            print(df.head(10).to_string(index=False))

    # -----------------------
    # 4) Execute chart code
    # -----------------------
    print("\n" + "="*40)
    print("CHART")
    print("="*40)

    if not isinstance(chart_code, str) or not chart_code.strip():
        print("<no chart code found>")
        return

    print("Chart code preview:\n")
    print(chart_code[:800])

    # Clean chart code (remove imports and dangerous lines)
    cleaned = []
    for ln in chart_code.splitlines():
        if re.match(r"^\s*(import\s+|from\s+)", ln):
            continue
        if any(danger in ln for danger in ["open(", "os.", "sys.", "subprocess"]):
            continue
        cleaned.append(ln)

    cleaned_code = "\n".join(cleaned).strip()

    # Validate chart uses df and produces fig
    if "df" not in cleaned_code:
        print("Chart code does not reference df → cannot execute.")
        return

    if "fig" not in cleaned_code or "fig.show" not in cleaned_code:
        print("Chart code does not build fig.show() → cannot execute.")
        return

    if df is None or df.empty:
        print("Dataframe missing or empty → chart cannot execute.")
        return

    # Execute safely
    local_env = {"pd": pd, "px": px, "go": go, "df": df.copy()}
    try:
        exec(compile(cleaned_code, "<chart>", "exec"), local_env)
        fig = local_env.get("fig")
        if fig is not None:
            fig.show()
            print("Chart displayed successfully.")
        else:
            print("Chart code executed but did not create fig.")
    except Exception as e:
        print("Error executing chart code:", e)


In [0]:
core = result.get("chart_agent", result)
analysis = core.get("analysis_text") or core.get("analysis") or None
print("\n" + "="*40)
print("DATA ANALYSIS")
print("="*40)
print(analysis if analysis else "<no analysis text found>")

In [0]:
sql = core.get("generated_sql") or core.get("sql") or None
print("\n" + "="*40)
print("GENERATED SQL")
print("="*40)
print(sql if sql else "<no generated SQL found>")

In [0]:
df_obj = core.get("dataframe")

In [0]:
df = None
df_obj = core.get("dataframe")
if isinstance(df_obj, pd.DataFrame):
    df = df_obj

elif isinstance(df_obj, list):
    try:
        df = pd.DataFrame(df_obj)
    except Exception:
        df = None

elif isinstance(df_obj, dict):
    if "rows" in df_obj and isinstance(df_obj["rows"], list):
        df = pd.DataFrame(df_obj["rows"])
    else:
        try:
            df = pd.DataFrame(df_obj)
        except Exception:
            df = None

# -----------------------
# Print dataframe
# -----------------------
print("\n" + "="*40)
print("DATAFRAME (top rows)")
print("="*40)

if df is None:
    print("<no dataframe found or could not convert>")
else:
    print(f"Shape: {df.shape}")
    try:
        from IPython.display import display
        display(df.head(10))
    except Exception:
        print(df.head(10).to_string(index=False))

In [0]:
print(result)

In [0]:
result = agent.run("Show total SPENDS and total HH_GRPS for DOG species on weekly basis")
out = present(result)


In [0]:
import json
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from typing import Any, Dict

def parse_and_present(agent_result: Any, show_improved_chart: bool = False):
    """
    Parse agent_result (dict or JSON string) and present:
      - Analysis text (if present or fallback)
      - Generated SQL
      - Dataframe (head)
      - Execute & show agent chart code (if present)
    
    Parameters:
      - agent_result: dict or JSON string produced by your agent (the structure you pasted).
      - show_improved_chart: if True, renders an improved chart (sorted, labels). Only applied if agent produced the expected columns.
    """
    # Normalize input to dict
    if isinstance(agent_result, str):
        try:
            result = json.loads(agent_result)
        except Exception:
            raise ValueError("agent_result is a string but not valid JSON")
    elif isinstance(agent_result, dict):
        result = agent_result
    else:
        raise ValueError("agent_result must be a dict or JSON string")

    # Helper to safely pick keys (handles nested 'chart_agent' wrapper)
    def pick(k):
        # first check top-level
        if k in result:
            return result[k]
        # if wrapped like {'chart_agent': {...}}
        for pref in ("chart_agent", "chart", "agent", "result"):
            if pref in result and isinstance(result[pref], dict) and k in result[pref]:
                return result[pref][k]
        return None

    # 1) Analysis text
    analysis = pick("analysis_text") or pick("analysis") or pick("analysis_output") or pick("text_analysis")
    if isinstance(analysis, dict) and "text" in analysis:
        analysis = analysis["text"]
    print("\n" + "="*60)
    print("📊 DATA ANALYSIS")
    print("="*60)
    if analysis:
        print(analysis)
    else:
        print("<no analysis text found in result>")

    # 2) Generated SQL
    generated_sql = pick("generated_sql") or pick("sql") or pick("query") or pick("generated_query")
    print("\n" + "="*60)
    print("📝 GENERATED SQL")
    print("="*60)
    if generated_sql:
        print(generated_sql)
    else:
        print("<no generated SQL found>")

    # 3) Dataframe: support pandas.DataFrame, list-of-dicts, or JSON string
    df_obj = pick("dataframe") or pick("df") or pick("rows") or pick("data")
    df = None
    if df_obj is None:
        print("\n" + "="*60)
        print("📁 DATAFRAME")
        print("="*60)
        print("<no dataframe found in result>")
    else:
        # If it's already a DataFrame
        if isinstance(df_obj, pd.DataFrame):
            df = df_obj
        else:
            # if it's a list of dicts or a dict with 'rows'
            if isinstance(df_obj, list):
                try:
                    df = pd.DataFrame(df_obj)
                except Exception:
                    df = None
            elif isinstance(df_obj, dict):
                # maybe { '0': {..}, '1': {..} } or {"columns":..., "rows":...}
                if "rows" in df_obj and isinstance(df_obj["rows"], list):
                    df = pd.DataFrame(df_obj["rows"])
                elif "columns" in df_obj and "rows" in df_obj:
                    try:
                        df = pd.DataFrame(df_obj["rows"], columns=df_obj["columns"])
                    except Exception:
                        df = pd.DataFrame(df_obj["rows"])
                else:
                    # attempt to coerce dict-of-records
                    try:
                        df = pd.json_normalize(df_obj)
                    except Exception:
                        df = None
            elif isinstance(df_obj, str):
                # try parse json string
                try:
                    parsed = json.loads(df_obj)
                    if isinstance(parsed, list):
                        df = pd.DataFrame(parsed)
                    elif isinstance(parsed, dict) and "rows" in parsed:
                        df = pd.DataFrame(parsed["rows"])
                except Exception:
                    df = None

        print("\n" + "="*60)
        print("📁 DATAFRAME (top rows)")
        print("="*60)
        if df is None:
            print("<could not coerce dataframe into pandas.DataFrame>")
        else:
            # show shape and head
            print(f"Shape: {df.shape}")
            try:
                # in notebook env prefer display()
                try:
                    from IPython.display import display
                    display(df.head(10))
                except Exception:
                    print(df.head(10).to_string(index=False))
            except Exception as e:
                print("Error displaying dataframe:", e)

    # 4) Chart code and execution
    chart_code = pick("chart_code") or pick("chart") or pick("chart_code_snippet") or pick("chart_code_plain")
    print("\n" + "="*60)
    print("📈 CHART")
    print("="*60)
    if not chart_code:
        print("<no chart code found in result>")
        return {"analysis": analysis, "sql": generated_sql, "df": df, "chart_executed": False}

    print("Agent-generated chart code (preview):\n")
    preview = chart_code if len(chart_code) < 1000 else chart_code[:1000] + "\n... (truncated)"
    print(preview)

    # sanitize: remove import statements and disallowed constructs (basic)
    def sanitize_code(code: str) -> str:
        lines = []
        for ln in code.splitlines():
            if re.match(r"^\s*(import\s+\w+|from\s+\w+\s+import\s+)", ln):
                continue
            # avoid os/system calls (very basic heuristic)
            if re.search(r"\b(os\.|subprocess\.|sys\.|open\()", ln):
                continue
            lines.append(ln)
        return "\n".join(lines)

    cleaned = sanitize_code(chart_code)

    # Quick static checks
    if "df" not in cleaned:
        print("\n❌ Chart code does not reference 'df' — refusing to execute.")
        return {"analysis": analysis, "sql": generated_sql, "df": df, "chart_executed": False}

    if "fig" not in cleaned or "fig.show" not in cleaned:
        print("\n❌ Chart code does not create 'fig' and call 'fig.show()' — refusing to execute.")
        return {"analysis": analysis, "sql": generated_sql, "df": df, "chart_executed": False}

    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        print("\n❌ No valid dataframe available to render the chart.")
        return {"analysis": analysis, "sql": generated_sql, "df": df, "chart_executed": False}

    # Prepare safe env
    local_env = {
        "pd": pd,
        "px": px,
        "go": go,
        "df": df.copy(),
        "__builtins__": {"len": len, "range": range, "min": min, "max": max, "sum": sum}
    }

    # Execute chart
    try:
        compiled = compile(cleaned, "<chart_exec>", "exec")
        exec(compiled, local_env)
        fig = local_env.get("fig")
        if fig is None:
            print("\n❌ Chart code executed but did not create 'fig'.")
            return {"analysis": analysis, "sql": generated_sql, "df": df, "chart_executed": False}
        try:
            fig.show()
            print("\n✅ Chart executed and displayed.")
            chart_executed = True
        except Exception:
            # try fallback display
            try:
                from IPython.display import display
                display(fig)
                print("\n✅ Chart displayed via IPython.display.")
                chart_executed = True
            except Exception as e:
                print("\n⚠️ Chart created but could not be displayed in this environment:", e)
                chart_executed = False
    except Exception as e:
        print("\n❌ Error executing chart code:", e)
        chart_executed = False

    # Optionally show improved chart if requested (and if columns exist)
    if show_improved_chart and df is not None and isinstance(df, pd.DataFrame):
        if set(["BRAND", "TOTAL_SPENDS", "TOTAL_HH_GRPS"]).issubset(set(df.columns)):
            try:
                print("\n" + "="*60)
                print("🔧 IMPROVED CHART (sorted + labels)")
                print("="*60)
                plot_df = df.copy().sort_values("TOTAL_SPENDS", ascending=False)
                fig2 = px.bar(
                    plot_df,
                    x="BRAND",
                    y=["TOTAL_SPENDS", "TOTAL_HH_GRPS"],
                    title="DOG Species Spend — Brand-wise (sorted)",
                    barmode="group",
                    labels={"value":"Amount","variable":"Metric"}
                )
                fig2.update_layout(xaxis_title='Brand', yaxis_title='Value', legend_title='Metric', autosize=True)
                for i in range(min(2, len(fig2.data))):
                    fig2.data[i].text = plot_df[ ["TOTAL_SPENDS","TOTAL_HH_GRPS"][i] ]
                    fig2.data[i].textposition = 'outside'
                fig2.show()
            except Exception as e:
                print("Failed to render improved chart:", e)

    return {"analysis": analysis, "sql": generated_sql, "df": df, "chart_executed": chart_executed}

# ---------------------------
# Example usage with your pasted object:
# ---------------------------
# Suppose `agent_result_obj` is the Python dict you pasted earlier (the one that has 'chart_agent' key).
# You can call:
#
# out = parse_and_present(agent_result_obj, show_improved_chart=True)
#
# The function prints the formatted outputs and attempts to display the chart(s).


In [0]:
def run_agent_and_display_chart(prompt: str):
    print("\n" + "="*70)
    print("📊 CHART GENERATION")
    print("="*70)
    print(f"❓ Prompt: {prompt}\n")

    print("🔄 Running agent pipeline...")
    print("   - SQL Generation")
    print("   - Data Analysis")
    print("   - Chart Generation\n")

    try:
        # Run agent pipeline
        result = agent.run(prompt)

        if result:
            print("✅ Agent execution completed!\n")

            # Extract results
            chart_code = result.get('chart_code')
            dataframe = result.get('dataframe')
            generated_sql = result.get('generated_sql')
            analysis_text = result.get('analysis_text')

            # Display SQL
            print("📝 GENERATED SQL:")
            print("-" * 70)
            print(generated_sql if generated_sql else "N/A")
            print("-" * 70 + "\n")

            # Display data analysis
            print("📊 DATA ANALYSIS:")
            print("-" * 70)
            print(analysis_text if analysis_text else "N/A")
            print("-" * 70 + "\n")

            # Execute and display chart
            if chart_code and dataframe is not None and not dataframe.empty:
                print("📉 GENERATING CHART...")
                print("-" * 70)

                try:
                    # Clean chart code (remove any import statements)
                    cleaned_lines = [ln for ln in chart_code.split('\n')
                                     if not re.match(r'^\s*(import|from)\s+', ln)]
                    cleaned_code = '\n'.join(cleaned_lines)

                    # Prepare safe environment for exec
                    safe_globals = {
                        'pd': pd,
                        'px': px,
                        'go': go,
                        'df': dataframe.copy(),
                    }

                    # Execute the visualization code
                    exec(cleaned_code, safe_globals, safe_globals)

                    # Retrieve the figure object from executed code
                    fig = safe_globals.get('fig')

                    if fig:
                        # Display the Plotly chart inline
                        fig.show()
                        print("\n✅ Chart displayed successfully!\n")

                        # Optional: Display data summary
                        print("📋 DATA SUMMARY:")
                        print("-" * 70)
                        print(f"Rows: {len(dataframe)}")
                        print(f"Columns: {list(dataframe.columns)}")
                        print(f"Shape: {dataframe.shape}")
                        print("-" * 70)
                    else:
                        print("❌ No figure created from code")

                except Exception as e:
                    print(f"❌ Error executing chart code: {e}")
                    print("\nChart Code:")
                    print(chart_code)
            else:
                print("❌ No chart code or dataframe available")
        else:
            print("❌ No results from agent")

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
